In [15]:
# import libraries

import numpy as np
import pandas as pd
import geopandas as gpd

import json

import leafmap.maplibregl as leafmap

import os

Load files

In [16]:
# Load total pop
total_pop_folder = "Population Data"
pop_gdf = gpd.read_parquet(os.path.join(total_pop_folder, "swk_1km_2020_population_age_breakdown.parquet"))

# Load student pop
student_pop_gdf = gpd.read_parquet(os.path.join(total_pop_folder, "swk_1km_2020_population_stu_breakdown.parquet"))

# Load school pop
school_folder = "School Data"
school_gdf = gpd.read_parquet(os.path.join(school_folder, "swk_list_of_schools_2025.parquet"))

# Load sekolah daif (dillipitated schools)
sekolah_daif_folder = "School Data"
sekolah_daif_df = pd.read_csv(os.path.join(sekolah_daif_folder, "mys_sekolah_daif_data_jan26.csv"))

# Load combined routes
routes_folder = "OSRM Routes to Nearest School"
secondary_combined_routes_gdf = gpd.read_parquet(os.path.join(routes_folder, "secondary_combined_routes.parquet"))
primary_combined_routes_gdf = gpd.read_parquet(os.path.join(routes_folder, "primary_combined_routes.parquet"))

# Boundaries
boundary_folder = "Geographic Boundaries"
swk_districts_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_districts.geojson")) # District boundaries
swk_parlimen_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_parliament.geojson")) # Parlimen boundaries
swk_dun_gdf = gpd.read_file(os.path.join(boundary_folder, "sarawak_dun.geojson")) # DUN boundaries

In [17]:
# Create total, primary and secondary population gdfs
total_pop_gdf = pop_gdf.copy()[["id","total_pop","x","y","geometry"]]
primary_pop_gdf = student_pop_gdf.copy()[["id","primary_school_students","x","y","geometry"]]
secondary_pop_gdf = student_pop_gdf.copy()[["id","secondary_school_students","x","y","geometry"]]

# Round numbders
total_pop_gdf["total_pop"] = total_pop_gdf["total_pop"].round(0).astype(int)
primary_pop_gdf["primary_school_students"] = primary_pop_gdf["primary_school_students"].round(0).astype(int)
secondary_pop_gdf["secondary_school_students"] = secondary_pop_gdf["secondary_school_students"].round(0).astype(int)

# Rename columns
total_pop_gdf = total_pop_gdf.rename(columns={"id":"pop_id","total_pop":"total_population"})
primary_pop_gdf = primary_pop_gdf.rename(columns={"id":"pop_id"})
secondary_pop_gdf = secondary_pop_gdf.rename(columns={"id":"pop_id"})

In [18]:
# Simplify geometry
tolerance = 0.00001  # ~1 metre, adjust if needed

primary_combined_routes_gdf["geometry"] = (
    primary_combined_routes_gdf["geometry"]
    .simplify(tolerance, preserve_topology=True)
)

secondary_combined_routes_gdf["geometry"] = (
    secondary_combined_routes_gdf["geometry"]
    .simplify(tolerance, preserve_topology=True)
)


In [19]:
# Clean routes data
def clean_combined_routes_gdf(combined_routes_gdf):
    # Remove columns
    combined_routes_gdf = combined_routes_gdf.drop(columns=["euclidean_km"])
    
    # Rename columns
    combined_routes_gdf = combined_routes_gdf.rename(columns={
        "combined_route": "travel_mode",
        "osrm_km": "dist_km",
        "osrm_min": "time_min"
    })
    
    # Round numbers
    combined_routes_gdf["dist_km"] = combined_routes_gdf["dist_km"].round(1)
    combined_routes_gdf["time_min"] = combined_routes_gdf["time_min"].round(0)
    return combined_routes_gdf

secondary_combined_routes_gdf = clean_combined_routes_gdf(secondary_combined_routes_gdf)
primary_combined_routes_gdf = clean_combined_routes_gdf(primary_combined_routes_gdf)

In [20]:
# merge population gdf with combined routes gdf
secondary_pop_routes_gdf = secondary_pop_gdf[["pop_id","secondary_school_students"]].merge(
    secondary_combined_routes_gdf,
    on="pop_id",
    how="left"
)

primary_pop_routes_gdf = primary_pop_gdf[["pop_id","primary_school_students"]].merge(
    primary_combined_routes_gdf,
    on="pop_id",
    how="left"
)

In [21]:
# Clean boundaries data

# Districts
swk_districts_gdf = swk_districts_gdf[["name","geometry"]].rename(columns={"name":"district"})

# DUNs
swk_dun_gdf = swk_dun_gdf[["dun","geometry"]]
swk_parlimen_gdf = swk_parlimen_gdf[["parlimen","geometry"]]

Output file paths

In [22]:
output_folder_maps = "Interactive Maps"

Create school data joined with sekolah daif data

In [23]:
# Filter and clean sekolah daif data
status = ["BINA","PRABINA"]

sekolah_daif_df_f = sekolah_daif_df[sekolah_daif_df['NEGERI'] == 'SARAWAK'].copy()
sekolah_daif_df_f = sekolah_daif_df_f[sekolah_daif_df_f['STATUS'].isin(status)].copy()
sekolah_daif_df_f = sekolah_daif_df_f[["KOD SEKOLAH","TAHUN LULUS","PERUNTUKAN","NAMA PROJEK","STATUS","% JADUAL **", "% SEBENAR **"]]

sekolah_daif_df_f = sekolah_daif_df_f.rename(columns={
    "KOD SEKOLAH":"kod_sekolah",
    "TAHUN LULUS":"tahun_projek_daif_lulus",
    "PERUNTUKAN":"peruntukan_projek_daif",
    "NAMA PROJEK":"nama_projek_daif",
    "STATUS":"status",
    "% JADUAL **":"pct_jadual",
    "% SEBENAR **":"pct_sebenar"
})

sekolah_daif_df_f["sekolah_daif"] = "YA"

In [24]:
# Merge sekolah daif data
school_daif_gdf = school_gdf.merge(sekolah_daif_df_f, how="left", on="kod_sekolah")
school_daif_gdf["sekolah_daif"] = school_daif_gdf["sekolah_daif"].fillna("TIDAK")

In [25]:
# Create primary and school gdf
primary_school_gdf = school_daif_gdf.copy()[school_gdf["primary_secondary"]=="Primary"][["id","nama_sekolah","bil_murid","bil_guru","sekolah_daif","peruntukan_projek_daif","tahun_projek_daif_lulus","nama_projek_daif","status","pct_jadual","pct_sebenar","geometry"]]
secondary_school_gdf = school_daif_gdf.copy()[school_gdf["primary_secondary"]=="Secondary"][["id","nama_sekolah","bil_murid","bil_guru","sekolah_daif","peruntukan_projek_daif","tahun_projek_daif_lulus","nama_projek_daif","status","pct_jadual","pct_sebenar","geometry"]]

Create population data which shows whether population point is likely to go to sekolah daif

In [26]:
# Add sekolah daif data into routes_gdf
primary_pop_daif_routes_gdf = primary_pop_routes_gdf.merge(primary_school_gdf[["id","sekolah_daif"]],left_on="school_id",right_on="id")
secondary_pop_daif_routes_gdf = secondary_pop_routes_gdf.merge(secondary_school_gdf[["id","sekolah_daif"]],left_on="school_id",right_on="id")

# Rearrange columns
cols = primary_pop_daif_routes_gdf.columns.tolist()[:-3] + ["sekolah_daif","geometry"]
primary_pop_daif_routes_gdf = primary_pop_daif_routes_gdf[cols]

cols = secondary_pop_daif_routes_gdf.columns.tolist()[:-3] + ["sekolah_daif","geometry"]
secondary_pop_daif_routes_gdf = secondary_pop_daif_routes_gdf[cols]

Create interactive map for primary school

In [27]:
# PRIMARY SCHOOL

# -----------------------------
# 0. Define output folders
# -----------------------------

base_output_folder = output_folder_maps   # your original output folder
assets_folder = os.path.join(base_output_folder, "assets")
os.makedirs(assets_folder, exist_ok=True)

# HTML output file
output_path = os.path.join(
    base_output_folder,
    "swk_sekolah_daif_primary_interactive_map.html"
)

# -----------------------------
# 1. Create gdfs for pop and route layers
# -----------------------------

students = "primary_school_students"
route_gdf = primary_pop_daif_routes_gdf.rename(columns={"Travel Time":"time_category"}).copy()

# Define helper function
def create_layer_gdfs(category):
    gdf = route_gdf[route_gdf["sekolah_daif"]==category]
    gdf_route = gpd.GeoDataFrame(gdf.copy(), geometry="geometry", crs="EPSG:4326")
    gdf_pop = gpd.GeoDataFrame(
        gdf[["pop_id",students,"pop_x","pop_y","school_id","school_name","travel_mode","time_min","dist_km"]], 
        geometry=gpd.points_from_xy(gdf["pop_x"], gdf["pop_y"]), crs="EPSG:4326")
    
    return gdf_route, gdf_pop

gdf_route_daif, gdf_pop_daif = create_layer_gdfs("YA")
gdf_route_norm, gdf_pop_norm= create_layer_gdfs("TIDAK")

# -----------------------------
# 2. Create gdfs for school layers
# -----------------------------

school_gdf = primary_school_gdf.copy()

gdf_school_daif = school_gdf[school_gdf["sekolah_daif"]=="YA"]
gdf_school_norm = school_gdf[school_gdf["sekolah_daif"]=="TIDAK"]

# -----------------------------
# 3. Build the interactive map
# -----------------------------

x = (route_gdf["pop_x"].max() + route_gdf["pop_x"].min()) / 2
y = (route_gdf["pop_y"].max() + route_gdf["pop_y"].min()) / 2

m = leafmap.Map(height="600px", center=[x, y], use_message_queue=True, style="street")

# Basemap
m.add_basemap("Esri.WorldImagery", before_id=m.first_symbol_layer_id, visible=True)
m.add_tile_layer(
    url="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    name="Esri.WorldImagery (Plain)",
    attribution="Esri World Imagery",
    visible=False
)

# -----------------------------
# 4. Create Population Data Layers
# -----------------------------

# --- Pop and Routes ---

# Normal Population Points
m.add_gdf(
    gdf=gdf_pop_norm,
    name="Population Points",
    layer_type="circle",
    paint={"circle-radius": 3, 
           "circle-color": "#000000"},
    visible=True
)

m.add_gdf(
    gdf=gdf_route_norm,
    name="Routes",
    layer_type="line",
    paint={"line-color": "#000000"},
    visible=False
)

# Population Points Attending Sekolah Daif
m.add_gdf(
    gdf=gdf_pop_daif,
    name="Population Points (Sekolah Daif)",
    layer_type="circle",
    paint={"circle-radius": 3, 
           "circle-color": "#d73027"},
    visible=True
)

m.add_gdf(
    gdf=gdf_route_daif,
    name="Routes (Sekolah Daif)",
    layer_type="line",
    paint={"line-color": "#d73027"},
    visible=True
)

# -----------------------------
# 5. Adminstrative Boundaries
# -----------------------------

# Adminstrative boundaries
m.add_gdf(
    gdf=swk_districts_gdf,
    name="District Boundaries",
    layer_type="line",
    paint={"line-color": "#ffffffeb"},
    visible=True
)

m.add_gdf(
    gdf=swk_parlimen_gdf,
    name="Parlimen Boundaries",
    layer_type="line",
    paint={"line-color": "#ffffffeb"},
    visible=False
)

m.add_gdf(
    gdf=swk_dun_gdf,
    name="DUN Boundaries",
    layer_type="line",
    paint={"line-color": "#ffffffeb"},
    visible=False
)

# -----------------------------
# 6. Add school layers
# -----------------------------

# -- Primary Schools ---
image_pri_norm = "Interactive Maps/assets/icons/college_blue.png"
image_pri_daif = "Interactive Maps/assets/icons/college_red_inverse.png"

# Load normal school data
primary_norm_geojson = json.loads(gdf_school_norm.to_json())
source_pri_norm = {"type": "geojson", "data": primary_norm_geojson}

# Normal schools
m.add_image("pri_school_norm", image_pri_norm)
m.add_source("point_pri_norm", {"type": "geojson", "data": json.loads(gdf_school_norm.to_json())})
m.add_popup("Primary Schools")
m.add_layer({
    "id": "Primary Schools",
    "type": "symbol",
    "source": "point_pri_norm",
    "layout": {"icon-image": "pri_school_norm", "icon-size": 0.08}
})

# Sekolah daif schools
m.add_image("pri_school_daif", image_pri_daif)
m.add_source("point_pri_daif", {"type": "geojson", "data": json.loads(gdf_school_daif.to_json())})
m.add_popup("Primary Schools (Sekolah Daif)")
m.add_layer({
    "id": "Primary Schools (Sekolah Daif)",
    "type": "symbol",
    "source": "point_pri_daif",
    "layout": {"icon-image": "pri_school_daif", "icon-size": 0.08}
})

# -----------------------------
# 7. Add legend
# -----------------------------

colour_map = {
    "Normal":    "#000000",   # light neutral grey
    "Sekolah Daif":"#d73027"    # deep red (critical)
}

m.add_legend(
    title = "Sekolah Daif",
    legend_dict = colour_map,
    position = "bottom-right"
)

# -----------------------------
# 8. Add map title / sources
# -----------------------------

m.add_text(
    "Locations of Sekolah Daif (Primary Schools)",
    position="top-left",
    font_size=12
)

# -----------------------------
# 9. Layer control
# -----------------------------

m.add_layer_control(
    layer_ids=[
        "Population Points",
        "Population Points (Sekolah Daif)",
        "Primary Schools",
        "Primary Schools (Sekolah Daif)",
        "Routes",
        "Routes (Sekolah Daif)",
        "District Boundaries",
        "Parlimen Boundaries",
        "DUN Boundaries",
        "Esri.WorldImagery",
        "Esri.WorldImagery (Plain)"
    ],
    position="top-left"
)

# -----------------------------
# 10. Export HTML
# -----------------------------

m.to_html(
    output=output_path,
    title="Locations of Sekolah Daif (Primary Schools)",
    overwrite=True
)

m

OSError: [Errno 22] Invalid argument: 'Interactive Maps\\swk_sekolah_daif_primary_interactive_map.html'

Create interactive map for secondary school

In [ ]:
# SECONDARY SCHOOL

# -----------------------------
# 0. Define output folders
# -----------------------------

base_output_folder = output_folder_maps   # your original output folder
assets_folder = os.path.join(base_output_folder, "assets")
os.makedirs(assets_folder, exist_ok=True)

# HTML output file
output_path = os.path.join(
    base_output_folder,
    "swk_sekolah_daif_secondary_interactive_map.html"
)

# -----------------------------
# 1. Create gdfs for pop and route layers
# -----------------------------

students = "secondary_school_students"
route_gdf = secondary_pop_daif_routes_gdf.rename(columns={"Travel Time":"time_category"}).copy()

# Define helper function
def create_layer_gdfs(category):
    gdf = route_gdf[route_gdf["sekolah_daif"]==category]
    gdf_route = gpd.GeoDataFrame(gdf.copy(), geometry="geometry", crs="EPSG:4326")
    gdf_pop = gpd.GeoDataFrame(
        gdf[["pop_id",students,"pop_x","pop_y","school_id","school_name","travel_mode","time_min","dist_km"]], 
        geometry=gpd.points_from_xy(gdf["pop_x"], gdf["pop_y"]), crs="EPSG:4326")
    
    return gdf_route, gdf_pop

gdf_route_daif, gdf_pop_daif = create_layer_gdfs("YA")
gdf_route_norm, gdf_pop_norm= create_layer_gdfs("TIDAK")

# -----------------------------
# 2. Create gdfs for school layers
# -----------------------------

school_gdf = secondary_school_gdf.copy()

gdf_school_daif = school_gdf[school_gdf["sekolah_daif"]=="YA"]
gdf_school_norm = school_gdf[school_gdf["sekolah_daif"]=="TIDAK"]

# -----------------------------
# 3. Build the interactive map
# -----------------------------

x = (route_gdf["pop_x"].max() + route_gdf["pop_x"].min()) / 2
y = (route_gdf["pop_y"].max() + route_gdf["pop_y"].min()) / 2

m = leafmap.Map(height="600px", center=[x, y], use_message_queue=True, style="street")

# Basemap
m.add_basemap("Esri.WorldImagery", before_id=m.first_symbol_layer_id, visible=True)
m.add_tile_layer(
    url="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    name="Esri.WorldImagery (Plain)",
    attribution="Esri World Imagery",
    visible=False
)

# -----------------------------
# 4. Create Population Data Layers
# -----------------------------

# --- Pop and Routes ---

# Normal Population Points
m.add_gdf(
    gdf=gdf_pop_norm,
    name="Population Points",
    layer_type="circle",
    paint={"circle-radius": 3, 
           "circle-color": "#000000"},
    visible=True
)

m.add_gdf(
    gdf=gdf_route_norm,
    name="Routes",
    layer_type="line",
    paint={"line-color": "#000000"},
    visible=False
)

# Population Points Attending Sekolah Daif
m.add_gdf(
    gdf=gdf_pop_daif,
    name="Population Points (Sekolah Daif)",
    layer_type="circle",
    paint={"circle-radius": 3, 
           "circle-color": "#d73027"},
    visible=True
)

m.add_gdf(
    gdf=gdf_route_daif,
    name="Routes (Sekolah Daif)",
    layer_type="line",
    paint={"line-color": "#d73027"},
    visible=True
)

# -----------------------------
# 5. Adminstrative Boundaries
# -----------------------------

# Adminstrative boundaries
m.add_gdf(
    gdf=swk_districts_gdf,
    name="District Boundaries",
    layer_type="line",
    paint={"line-color": "#ffffffeb"},
    visible=True
)

m.add_gdf(
    gdf=swk_parlimen_gdf,
    name="Parlimen Boundaries",
    layer_type="line",
    paint={"line-color": "#ffffffeb"},
    visible=False
)

m.add_gdf(
    gdf=swk_dun_gdf,
    name="DUN Boundaries",
    layer_type="line",
    paint={"line-color": "#ffffffeb"},
    visible=False
)

# -----------------------------
# 6. Add school layers
# -----------------------------

# -- Secondary Schools ---
image_sec_norm = "Interactive Maps/assets/icons/college_orange.png"
image_sec_daif = "Interactive Maps/assets/icons/college_red_inverse.png"

# Load normal school data
sec_norm_geojson = json.loads(gdf_school_norm.to_json())
source_sec_norm = {"type": "geojson", "data": sec_norm_geojson}

# Normal schools
m.add_image("sec_school_norm", image_sec_norm)
m.add_source("point_sec_norm", {"type": "geojson", "data": json.loads(gdf_school_norm.to_json())})
m.add_popup("Secondary Schools")
m.add_layer({
    "id": "Secondary Schools",
    "type": "symbol",
    "source": "point_sec_norm",
    "layout": {"icon-image": "sec_school_norm", "icon-size": 0.08}
})

# Sekolah daif schools
m.add_image("sec_school_daif", image_sec_daif)
m.add_source("point_sec_daif", {"type": "geojson", "data": json.loads(gdf_school_daif.to_json())})
m.add_popup("Secondary Schools (Sekolah Daif)")
m.add_layer({
    "id": "Secondary Schools (Sekolah Daif)",
    "type": "symbol",
    "source": "point_sec_daif",
    "layout": {"icon-image": "sec_school_daif", "icon-size": 0.08}
})

# -----------------------------
# 7. Add legend
# -----------------------------

colour_map = {
    "Normal":    "#000000",   # light neutral grey
    "Sekolah Daif":"#d73027"    # deep red (critical)
}

m.add_legend(
    title = "Sekolah Daif",
    legend_dict = colour_map,
    position = "bottom-right"
)

# -----------------------------
# 8. Add map title / sources
# -----------------------------

m.add_text(
    "Locations of Sekolah Daif (Primary Schools)",
    position="top-left",
    font_size=12
)

# -----------------------------
# 9. Layer control
# -----------------------------

m.add_layer_control(
    layer_ids=[
        "Population Points",
        "Population Points (Sekolah Daif)",
        "Secondary Schools",
        "Secondary Schools (Sekolah Daif)",
        "Routes",
        "Routes (Sekolah Daif)",
        "District Boundaries",
        "Parlimen Boundaries",
        "DUN Boundaries",
        "Esri.WorldImagery",
        "Esri.WorldImagery (Plain)"
    ],
    position="top-left"
)

# -----------------------------
# 10. Export HTML
# -----------------------------

m.to_html(
    output=output_path,
    title="Locations of Sekolah Daif (Secondary Schools)",
    overwrite=True
)

m

Html(children=[<leafmap.maplibregl.Map object at 0x000001ACE48CF970>, Card(children=[Btn(children=[Icon(childr…